In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display

In [2]:
project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

raw_file = project_dir / "data" / "raw" / "2020-05-01" / "ted_talks_en.csv"

df = pd.read_csv(raw_file)

print("Original dataset loaded successfully.")
print("Original number of rows:", len(df))
print("Original number of columns:", len(df.columns))

display(df.head())

Original dataset loaded successfully.
Original number of rows: 4005
Original number of columns: 19


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre..."
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac..."
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun..."
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,en,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",219.0,1116,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,en,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",4931.0,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...


In [3]:
original_row_count = len(df)
original_column_count = len(df.columns)
original_duplicate_rows = int(df.duplicated().sum())
original_duplicate_ids = int(df["talk_id"].duplicated().sum())

print("Original rows:", original_row_count)
print("Original columns:", original_column_count)
print("Exact duplicate rows:", original_duplicate_rows)
print("Repeated talk IDs:", original_duplicate_ids)

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Count"))

Original rows: 4005
Original columns: 19
Exact duplicate rows: 0
Repeated talk IDs: 0

Missing values:


,Missing Count
talk_id,0
title,0
speaker_1,0
all_speakers,4
occupations,522
about_speakers,503
views,0
recorded_date,1
published_date,0
event,0


In [4]:
important_columns = [
    "title",
    "topics",
    "description",
    "transcript",
    "url",
    "speaker_1"
]

for column in important_columns:
    if column not in df.columns:
        print(f"{column}: Column not available")
        continue

    missing = int(df[column].isna().sum())

    empty = int(
        df[column]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(f"{column}: Missing = {missing}, Empty = {empty}")

title: Missing = 0, Empty = 0
topics: Missing = 0, Empty = 0
description: Missing = 0, Empty = 0
transcript: Missing = 0, Empty = 0
url: Missing = 0, Empty = 0
speaker_1: Missing = 0, Empty = 0


In [5]:
text_columns = [
    "title",
    "topics",
    "description",
    "transcript",
    "url",
    "speaker_1",
    "all_speakers",
    "occupations",
    "about_speakers"
]

available_text_columns = [
    column for column in text_columns if column in df.columns
]

for column in available_text_columns:
    df[column] = df[column].fillna("")

print("Missing values in selected text columns after handling:")

display(df[available_text_columns].isna().sum().to_frame("Missing Count"))

Missing values in selected text columns after handling:


,Missing Count
title,0
topics,0
description,0
transcript,0
url,0
speaker_1,0
all_speakers,0
occupations,0
about_speakers,0


In [6]:
exact_duplicate_count = int(df.duplicated().sum())

print("Exact duplicate rows before removal:", exact_duplicate_count)

Exact duplicate rows before removal: 0


In [7]:
df = df.drop_duplicates().copy()

print("Rows after removing exact duplicates:", len(df))
print("Exact duplicate rows remaining:", int(df.duplicated().sum()))

Rows after removing exact duplicates: 4005
Exact duplicate rows remaining: 0


In [8]:
duplicate_id_mask = df["talk_id"].duplicated(keep=False)

duplicate_id_records = df.loc[
    duplicate_id_mask,
    ["talk_id", "title", "url"]
]

print("Records with repeated talk IDs:", len(duplicate_id_records))
display(duplicate_id_records.head(20))

Records with repeated talk IDs: 0


,talk_id,title,url


In [9]:
if "url" in df.columns:
    valid_urls = df["url"].astype(str).str.strip()
    valid_url_mask = valid_urls.ne("")

    repeated_url_mask = (
        valid_url_mask
        & valid_urls.duplicated(keep=False)
    )

    duplicate_url_records = df.loc[
        repeated_url_mask,
        ["talk_id", "title", "url"]
    ]

    print("Records with repeated non-empty URLs:",
          len(duplicate_url_records))

    display(duplicate_url_records.head(20))

Records with repeated non-empty URLs: 0


,talk_id,title,url


In [10]:
def clean_text(value):
    if pd.isna(value):
        return ""

    text = str(value).lower()

    # Replace line breaks and repeated whitespace with one space
    text = re.sub(r"\s+", " ", text)

    # Remove spaces at the beginning and end
    text = text.strip()

    return text

In [11]:
fields_to_clean = [
    "title",
    "topics",
    "description",
    "transcript"
]

for column in fields_to_clean:
    if column in df.columns:
        df[f"{column}_clean"] = df[column].apply(clean_text)

display(
    df[
        ["title", "title_clean", "description", "description_clean"]
    ].head()
)

,title,title_clean,description,description_clean
0,Averting the climate crisis,averting the climate crisis,With the same humor and humanity he exuded in ...,with the same humor and humanity he exuded in ...
1,The best stats you've ever seen,the best stats you've ever seen,You've never seen data presented like this. Wi...,you've never seen data presented like this. wi...
2,Simplicity sells,simplicity sells,New York Times columnist David Pogue takes aim...,new york times columnist david pogue takes aim...
3,Greening the ghetto,greening the ghetto,"In an emotionally charged talk, MacArthur-winn...","in an emotionally charged talk, macarthur-winn..."
4,Do schools kill creativity?,do schools kill creativity?,Sir Ken Robinson makes an entertaining and pro...,sir ken robinson makes an entertaining and pro...


In [12]:
comparison_columns = [
    "title",
    "title_clean",
    "description",
    "description_clean"
]

display(df[comparison_columns].head(10))

,title,title_clean,description,description_clean
0,Averting the climate crisis,averting the climate crisis,With the same humor and humanity he exuded in ...,with the same humor and humanity he exuded in ...
1,The best stats you've ever seen,the best stats you've ever seen,You've never seen data presented like this. Wi...,you've never seen data presented like this. wi...
2,Simplicity sells,simplicity sells,New York Times columnist David Pogue takes aim...,new york times columnist david pogue takes aim...
3,Greening the ghetto,greening the ghetto,"In an emotionally charged talk, MacArthur-winn...","in an emotionally charged talk, macarthur-winn..."
4,Do schools kill creativity?,do schools kill creativity?,Sir Ken Robinson makes an entertaining and pro...,sir ken robinson makes an entertaining and pro...
5,Behind the design of Seattle's library,behind the design of seattle's library,Architect Joshua Prince-Ramus takes the audien...,architect joshua prince-ramus takes the audien...
6,Letting go of God,letting go of god,When two young Mormon missionaries knock on Ju...,when two young mormon missionaries knock on ju...
7,Let's teach religion -- all religion -- in sch...,let's teach religion -- all religion -- in sch...,Philosopher Dan Dennett calls for religion -- ...,philosopher dan dennett calls for religion -- ...
8,A life of purpose,a life of purpose,"Pastor Rick Warren, author of ""The Purpose-Dri...","pastor rick warren, author of ""the purpose-dri..."
9,My wish: A global day of film,my wish: a global day of film,Jehane Noujaim unveils her 2006 TED Prize wish...,jehane noujaim unveils her 2006 ted prize wish...


In [13]:
combined_columns = [
    "title_clean",
    "topics_clean",
    "description_clean",
    "transcript_clean"
]

available_combined_columns = [
    column for column in combined_columns if column in df.columns
]

df["combined_text"] = (
    df[available_combined_columns]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Combined text created successfully.")

display(df[["title", "combined_text"]].head())

Combined text created successfully.


,title,combined_text
0,Averting the climate crisis,averting the climate crisis ['alternative ener...
1,The best stats you've ever seen,"the best stats you've ever seen ['africa', 'as..."
2,Simplicity sells,"simplicity sells ['computers', 'entertainment'..."
3,Greening the ghetto,"greening the ghetto ['macarthur grant', 'activ..."
4,Do schools kill creativity?,"do schools kill creativity? ['children', 'crea..."


In [14]:
empty_combined_count = int(
    df["combined_text"].fillna("").str.strip().eq("").sum()
)

print("Records with empty combined text:", empty_combined_count)
print("Total records available:", len(df))

Records with empty combined text: 0
Total records available: 4005


In [15]:
processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "ted_talks_cleaned.csv"

df.to_csv(output_file, index=False)

print("Cleaned dataset saved successfully.")
print("Output location:", output_file)
print("Final dataset shape:", df.shape)

Cleaned dataset saved successfully.
Output location: c:\Users\anish\ted-talks-recommender\data\processed\ted_talks_cleaned.csv
Final dataset shape: (4005, 24)


In [16]:
cleaned_df = pd.read_csv(output_file)

print("BEFORE CLEANING")
print("Rows:", original_row_count)
print("Columns:", original_column_count)
print("Exact duplicate rows:", original_duplicate_rows)
print("Repeated talk IDs:", original_duplicate_ids)

print("\nAFTER CLEANING")
print("Rows:", len(cleaned_df))
print("Columns:", len(cleaned_df.columns))
print("Exact duplicate rows:", int(cleaned_df.duplicated().sum()))
print("Repeated talk IDs:", int(cleaned_df["talk_id"].duplicated().sum()))

print("\nRequired column exists:", "combined_text" in cleaned_df.columns)
print("Saved dataset loaded successfully:", output_file.exists())

BEFORE CLEANING
Rows: 4005
Columns: 19
Exact duplicate rows: 0
Repeated talk IDs: 0

AFTER CLEANING
Rows: 4005
Columns: 24
Exact duplicate rows: 0
Repeated talk IDs: 0

Required column exists: True
Saved dataset loaded successfully: True


# Task 3: Data Cleaning and Text Preprocessing Report

## 1. Objective

The objective of this task was to clean the TED Talks dataset and prepare its text fields for a content-based recommendation system.

## 2. Missing-Value Handling

Missing values in selected text fields were replaced with empty strings in the working dataset. This allows the text fields to be combined consistently without introducing the words `nan` or `None`.

Talks were not removed simply because their transcripts were missing. Other available fields, such as the title, topics, or description, can still provide useful information.

## 3. Duplicate Handling

Exact duplicate rows were identified and removed. Repeated talk IDs and URLs were checked separately and should be investigated before any further records are removed.

## 4. Text Preprocessing

The preprocessing steps included:
- Converting selected text fields to lowercase.
- Normalizing repeated whitespace.
- Removing leading and trailing whitespace.
- Creating separate cleaned text columns for comparison.
- Combining the cleaned title, topics, description, and transcript into `combined_text`.

Punctuation was retained to avoid unnecessarily removing meaningful text.

## 5. Combined Text

The `combined_text` column combines available cleaned text fields into one text representation for each TED Talk. It will be used in the next stage for text vectorization and similarity-based recommendations.

## 6. Output

The processed dataset was saved to:

`data/processed/ted_talks_cleaned.csv`

The original dataset in `data/raw/` was preserved.

## 7. Limitations and Next Steps

Replacing missing text with empty strings does not recover the missing information. Repeated IDs or URLs require further investigation if any are found. Lowercase conversion and whitespace normalization do not perform advanced language processing.

The next stage will convert the combined text into numerical features and use those features to compare TED Talks for recommendations.